# Spark Setup

In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

HOST_IP = "192.168.64.1"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .getOrCreate()
)

Streaming Implementation

In [ ]:
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])

def read_topic(topic, source):
    return(
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{HOST_IP}:9092")
        .option("subscribe", topic)
        .load()
        .selectExpr("CAST(value AS STRING)")
        .withColumn("source", lit(source))
        .select(
            from_json(col("value"), event_schema).alias("data"),
            col("source")
        )
        .select("data.*", "source")
    )

# Read from all 3 producers
camera_a_stream = read_topic("camera-events-A", "camera-a")
camera_b_stream = read_topic("camera-events-B", "camera-b")
camera_c_stream = read_topic("camera-events-C", "camera-c")

# Combine the streams
combined_stream = camera_a_stream.union(camera_b_stream).union(camera_c_stream)

def log_combined_stream(batch_df, batch_id):

    now = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    row_count = batch_df.count()

    print("\n" + "=" * 70)
    print(f"[{now}] combined_stream processed")
    print(f"Spark Batch ID: {batch_id}")
    print(f"Rows received: {row_count}")
    print("=" * 70)

    batch_df.show(
        truncate=False
    )


# stream_logger = (
#     combined_stream
#     .writeStream
#     .foreachBatch(log_combined_stream)
#     .outputMode("append")
#     .start()
# )

# stream_logger.awaitTermination()

camera_df = spark.read.csv(
    f"{Path('..')}/data/camera.csv",
    header=True,
    inferSchema=True
)

instant_violations = (
    combined_stream.join(
    camera_df, "camera_id"
    )
    .filter(
        (col("speed_reading") > col("speed_limit"))
    )
)

def log_instant_violations(instant_violations_df, batch_id):

    now = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    row_count = instant_violations_df.count()

    print("\n" + "=" * 70)
    print(f"[{now}] instant_violations processed")
    print(f"Spark Batch ID: {batch_id}")
    print(f"Rows received: {row_count}")
    print("=" * 70)

    instant_violations_df.show(
        truncate=False
    )

query = (
    instant_violations.writeStream
    .foreachBatch(log_instant_violations)
    .outputMode("append")
    .option("truncate", "false")
    .start()
)

query.awaitTermination()



[2026-05-11 07:51:42] instant_violations processed
Spark Batch ID: 0
Rows received: 0
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+
|camera_id|event_id|batch_id|car_plate|timestamp|speed_reading|source|latitude|longitude|position|speed_limit|
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+


[2026-05-11 07:51:42] instant_violations processed
Spark Batch ID: 1
Rows received: 24
+---------+------------------------------------+--------+---------+-------------------+-------------+--------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp          |speed_reading|source  |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+-


[2026-05-11 07:51:48] instant_violations processed
Spark Batch ID: 4
Rows received: 1
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp                 |speed_reading|source  |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|2        |eb9614ad-ee22-4001-835f-4242e362be01|8       |JEJ 6    |2024-01-01T08:08:34.116409|111.2        |camera-b|2.162418757|102.6524549|153.5   |110        |
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+


[2026-05-11 07:51:49] instant_violations processed
Spark Batch ID: 5
Rows received: 0
+---------


[2026-05-11 07:51:59] instant_violations processed
Spark Batch ID: 11
Rows received: 2
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp                 |speed_reading|source  |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|3        |2f38e970-5042-4d91-8ef3-f8e4bed9908f|10      |CZ 592   |2024-01-01T08:00:47.453301|161.4        |camera-c|2.167352891|102.6449144|154.5   |90         |
|3        |fa9e3b2a-de4e-48be-b97b-1c01e95e437a|10      |AH 8     |2024-01-01T08:00:49.247531|165.5        |camera-c|2.167352891|102.6449144|154.5   |90         |
+---------+------------------------------------+--------+---------+--------------------------+---


[2026-05-11 07:52:09] instant_violations processed
Spark Batch ID: 17
Rows received: 1
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp                 |speed_reading|source  |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|3        |033183dc-45d4-4cab-98cf-a4f05088c76b|12      |ZPG 90   |2024-01-01T08:01:16.062056|101.4        |camera-c|2.167352891|102.6449144|154.5   |90         |
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+


[2026-05-11 07:52:10] instant_violations processed
Spark Batch ID: 18
Rows received: 11
+------


[2026-05-11 07:52:19] instant_violations processed
Spark Batch ID: 23
Rows received: 1
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp                 |speed_reading|source  |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|3        |dfc12ef9-a6d4-48ab-8dab-47eae3e90e1c|14      |IFY 23   |2024-01-01T08:09:12.383098|115.7        |camera-c|2.167352891|102.6449144|154.5   |90         |
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+


[2026-05-11 07:52:20] instant_violations processed
Spark Batch ID: 24
Rows received: 11
+------

...

+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp                 |speed_reading|source  |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|2        |def44cb3-9553-4637-a9dc-5c33c6c2eab6|63      |GNU 052  |2024-01-01T08:35:55.800945|123.3        |camera-b|2.162418757|102.6524549|153.5   |110        |
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+


[2026-05-11 07:56:24] instant_violations processed
Spark Batch ID: 170
Rows received: 0
+---------+--------+--------+---------+---------+-------------+------+--------+---------+------


[2026-05-11 07:56:33] instant_violations processed
Spark Batch ID: 175
Rows received: 0
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+
|camera_id|event_id|batch_id|car_plate|timestamp|speed_reading|source|latitude|longitude|position|speed_limit|
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+


[2026-05-11 07:56:34] instant_violations processed
Spark Batch ID: 176
Rows received: 1
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp                 |speed_reading|source  |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+---

+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+
|camera_id|event_id|batch_id|car_plate|timestamp|speed_reading|source|latitude|longitude|position|speed_limit|
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+


[2026-05-11 07:56:45] instant_violations processed
Spark Batch ID: 182
Rows received: 0
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+
|camera_id|event_id|batch_id|car_plate|timestamp|speed_reading|source|latitude|longitude|position|speed_limit|
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+
+---------+--------+--------+---------+---------+-------------+------+--------+---------+--------+-----------+


[2026-05-11 07:56:46

+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp                 |speed_reading|source  |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|3        |9b85a771-6ae5-4b31-9c86-78131ec2ae88|69      |YE 6517  |2024-01-01T08:26:20.939769|119.7        |camera-c|2.167352891|102.6449144|154.5   |90         |
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+


[2026-05-11 07:56:56] instant_violations processed
Spark Batch ID: 189
Rows received: 9
+---------+------------------------------------+--------+---------+-------------------+--------


[2026-05-11 07:57:05] instant_violations processed
Spark Batch ID: 194
Rows received: 1
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|camera_id|event_id                            |batch_id|car_plate|timestamp                 |speed_reading|source  |latitude   |longitude  |position|speed_limit|
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+
|3        |835c8eb6-6f3c-4675-8b7d-3fba15c58d78|71      |PW 7     |2024-01-01T08:26:49.807145|90.2         |camera-c|2.167352891|102.6449144|154.5   |90         |
+---------+------------------------------------+--------+---------+--------------------------+-------------+--------+-----------+-----------+--------+-----------+


[2026-05-11 07:57:06] instant_violations processed
Spark Batch ID: 195
Rows received: 6
